In [ ]:
!nvidia-smi
!pip install -q transformers accelerate bitsandbytes huggingface_hub

In [ ]:
import json
from pathlib import Path
import subprocess
import threading
import time
import pandas as pd
from google.colab import

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

In [ ]:
base_dir = Path("/content/drive/MyDrive/") # adjust

zero_shot_path = base_dir / "zero_shot.json"
few_shot_path = base_dir / "few_shot.json"
pr_experiment_path = base_dir / "pr_experiment_data.jsonl"

print("zero_shot.json:         ", zero_shot_path)
print("pr_experiment_data.jsonl:", pr_experiment_path)


def load_prompt_json(path):
    """Load JSON that contains either:
       - a dict with 'prompt'
       - a list of dicts with 'prompt'
       Return: list of prompt strings.
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict) and "prompt" in data:
        return [data["prompt"]]
    elif isinstance(data, list):
        prompts = []
        for item in data:
            if isinstance(item, dict) and "prompt" in item:
                prompts.append(item["prompt"])
        return prompts
    else:
        raise ValueError(f"Unexpected JSON structure in {path}")


In [ ]:
# Load zero-shot prompts (template with {{context}} and {{diff_hunk}})
zero_shot_prompts = load_prompt_json(zero_shot_path)
zero_template = zero_shot_prompts[0]

few_shot_prompts = load_prompt_json(few_shot_path)
few_template = few_shot_prompts[0]

print("=== Zero-shot template ===")
print(zero_template)

# Load JSONL, take first example
pr_experiment_data = []
with open(pr_experiment_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        pr_experiment_data.append(json.loads(line))

if not pr_experiment_data:
    raise RuntimeError("pr_experiment_data.jsonl is empty!")

first_example = pr_experiment_data[9]

print("\n=== First JSONL example attributes ===")
print(list(first_example.keys()))

print("\n=== First JSONL example preview ===")
for k, v in first_example.items():
    print(f"{k}: {repr(v)[:200]}")

# Extract context and diff_hunk
context_value = first_example["context"]
diff_hunk_value = first_example["diff_hunk"]

# Fill the template
filled_prompt = (
    zero_template
    .replace("{{context}}", context_value)
    .replace("{{diff_hunk}}", diff_hunk_value)
)

print("\n=== Filled prompt (what we'll send to Phi) ===")
print(filled_prompt[:2000])  # truncate just in case


In [ ]:
from huggingface_hub import login

# This opens a prompt where you paste your HF token.
# Make sure your token has access to Gemma (accept license on HF first).
login()

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


# Models

In [ ]:
phi_model_id = "microsoft/Phi-3-mini-4k-instruct"

phi_tokenizer = AutoTokenizer.from_pretrained(phi_model_id)
phi_model = AutoModelForCausalLM.from_pretrained(
    phi_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,  # or torch.float16 if needed
)

phi_model.eval()

print("Loaded Phi:", phi_model_id)


In [ ]:
gemma_model_id = "google/gemma-2-2b-it"

gemma_bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_model_id)
gemma_model = AutoModelForCausalLM.from_pretrained(
    gemma_model_id,
    quantization_config=gemma_bnb_config,
    device_map="auto",
)

gemma_model.eval()

print("Loaded Gemma:", gemma_model_id)


In [ ]:
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM

mistral_model_id = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

mistral_tokenizer = AutoTokenizer.from_pretrained(mistral_model_id)
mistral_model = AutoModelForCausalLM.from_pretrained(
    mistral_model_id,
    quantization_config=bnb_config,
    device_map="auto",
)


In [ ]:
class GPUPowerMonitor:
    """
    Simple GPU power logger using `nvidia-smi`.
    - Samples GPU power every `interval` seconds.
    - Estimates average power during the measured block.
    """

    def __init__(self, interval=0.2, gpu_index=0):
        self.interval = interval
        self.gpu_index = gpu_index
        self.samples = []
        self._running = False
        self._thread = None

    def _get_power_watts(self):
        try:
            out = subprocess.check_output([
                "nvidia-smi",
                f"--id={self.gpu_index}",
                "--query-gpu=power.draw",
                "--format=csv,noheader,nounits"
            ])
            val = out.decode("utf-8").strip()
            return float(val)
        except Exception:
            return None

    def _worker(self):
        while self._running:
            p = self._get_power_watts()
            if p is not None:
                self.samples.append(p)
            time.sleep(self.interval)

    def start(self):
        self.samples = []
        self._running = True
        self._thread = threading.Thread(target=self._worker, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread is not None:
            self._thread.join()
        self._thread = None

    def average_power(self):
        if not self.samples:
            return 0.0
        return sum(self.samples) / len(self.samples)


# Experiments

In [ ]:
def run_gpt4_with_metrics(prompt, max_new_tokens=256, model_name="gpt-4o"):
    import time as _time

    try:
        messages = [
            {"role": "user", "content": prompt}
        ]

        start = _time.perf_counter()
        response = client.chat.completions.create(
            model=model_name,          # e.g., "gpt-4o", "gpt-4.1-mini", etc.
            messages=messages,
            max_tokens=max_new_tokens,
            temperature=0.0,           # deterministic, like your local runs
        )
        end = _time.perf_counter()

        elapsed_s = end - start

        # Extract text
        response_text = (response.choices[0].message.content or "").strip()

        # Token usage (API gives this directly)
        usage = response.usage
        input_tokens = usage.prompt_tokens
        output_tokens = usage.completion_tokens
        total_tokens = input_tokens + output_tokens

        # We *cannot* measure GPU power/energy on OpenAI's servers
        avg_power_w = None
        energy_joules = None
        energy_wh = None

        print(f"=== SLM Response ({model_name}) ===")
        print(response_text)

        print(f"\n=== Metrics ({model_name}) ===")
        print(f"Elapsed time (s):        {elapsed_s:.3f}")
        print(f"Input tokens:            {input_tokens}")
        print(f"Output tokens:           {output_tokens}")
        print(f"Total tokens:            {total_tokens}")
        if elapsed_s > 0:
            print(f"Throughput (tokens/s):   {total_tokens / elapsed_s:.2f}")
        print(f"Avg GPU power (W):       {avg_power_w}")
        print(f"Energy (J, GPU only):    {energy_joules}")
        print(f"Energy (Wh, GPU only):   {energy_wh}")

        return response_text, {
            "elapsed_s": elapsed_s,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "avg_power_w": avg_power_w,
            "energy_joules": energy_joules,
            "energy_wh": energy_wh,
        }

    except Exception as e:
        print(f"=== ERROR during {model_name} generation ===")
        print(repr(e))

        return None, {
            "elapsed_s": None,
            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None,
            "avg_power_w": None,
            "energy_joules": None,
            "energy_wh": None,
        }


In [ ]:
def run_slm_with_metrics(
    prompt: str,
    model_key: str,
    max_new_tokens: int = 256,
):
    """
    Unified runner for local SLMs:
      - phi
      - mistral
      - gemma

    Keeps logic identical to the original per-model implementations.
    """

    import time as _time

    # -------- Model registry (must exist globally) --------
    registry = {
        "phi": {
            "model": phi_model,
            "tokenizer": phi_tokenizer,
            "needs_attention_mask": False,
            "pad_token_id": None,
            "label": "Phi",
        },
        "mistral": {
            "model": mistral_model,
            "tokenizer": mistral_tokenizer,
            "needs_attention_mask": True,
            "pad_token_id": mistral_tokenizer.eos_token_id,
            "label": "Mistral",
        },
        "gemma": {
            "model": gemma_model,
            "tokenizer": gemma_tokenizer,
            "needs_attention_mask": True,
            "pad_token_id": gemma_tokenizer.eos_token_id,
            "label": "Gemma",
        },
    }

    key = model_key.lower().strip()
    if key not in registry:
        raise ValueError(f"Unknown model_key='{model_key}'. Use one of {list(registry.keys())}")

    spec = registry[key]
    model = spec["model"]
    tokenizer = spec["tokenizer"]
    needs_attention_mask = spec["needs_attention_mask"]
    pad_token_id = spec["pad_token_id"]
    label = spec["label"]

    monitor = None

    try:
        # ---------------- Prompt → chat format ----------------
        messages = [{"role": "user", "content": prompt}]

        input_ids = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
        ).to(model.device)

        input_len = input_ids.shape[1]

        # Optional attention mask (Mistral / Gemma only)
        if needs_attention_mask:
            attention_mask = torch.ones_like(input_ids)
            inputs = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
            }
        else:
            inputs = {"input_ids": input_ids}

        # ---------------- Warm-up (unchanged) ----------------
        with torch.no_grad():
            _ = model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=False,
                **({"pad_token_id": pad_token_id} if pad_token_id is not None else {}),
            )
            torch.cuda.synchronize()

        # ---------------- Timed run + power ----------------
        monitor = GPUPowerMonitor(interval=0.2)
        monitor.start()
        start = _time.perf_counter()

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                **({"pad_token_id": pad_token_id} if pad_token_id is not None else {}),
            )
            torch.cuda.synchronize()

        end = _time.perf_counter()
        monitor.stop()

        # ---------------- Metrics ----------------
        elapsed_s = end - start
        avg_power_w = monitor.average_power()
        energy_joules = avg_power_w * elapsed_s
        energy_wh = avg_power_w * (elapsed_s / 3600.0)

        generated_ids = outputs[0][input_len:]
        response_text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True,
        ).strip()

        input_tokens = int(input_len)
        output_tokens = int(generated_ids.shape[0])
        total_tokens = input_tokens + output_tokens

        # ---------------- Logging ----------------
        print(f"=== SLM Response ({label}) ===")
        print(response_text)

        print(f"\n=== Metrics ({label}) ===")
        print(f"Elapsed time (s):        {elapsed_s:.3f}")
        print(f"Input tokens:            {input_tokens}")
        print(f"Output tokens:           {output_tokens}")
        print(f"Total tokens:            {total_tokens}")
        if elapsed_s > 0:
            print(f"Throughput (tokens/s):   {total_tokens / elapsed_s:.2f}")
        print(f"Avg GPU power (W):       {avg_power_w:.2f}")
        print(f"Energy (J, GPU only):    {energy_joules:.2f}")
        print(f"Energy (Wh, GPU only):   {energy_wh:.6f}")

        return response_text, {
            "elapsed_s": elapsed_s,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "avg_power_w": avg_power_w,
            "energy_joules": energy_joules,
            "energy_wh": energy_wh,
        }

    except Exception as e:
        try:
            if monitor is not None:
                monitor.stop()
        except Exception:
            pass

        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

        print(f"=== ERROR during {label} generation ===")
        print(repr(e))

        return None, {
            "elapsed_s": None,
            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None,
            "avg_power_w": None,
            "energy_joules": None,
            "energy_wh": None,
        }


In [ ]:
rows = []

for example in pr_experiment_data:
    # original fields from JSONL
    context = example.get("context", "")
    diff_hunk = example.get("diff_hunk", "")
    original_comment = example.get("comment", "")

    # build prompt for this example
    prompt = (
        zero_template
        .replace("{{context}}", context)
        .replace("{{diff_hunk}}", diff_hunk)
    )

    # run SLM (Phi) + collect metrics
    slm_response, metrics = run_slm_with_metrics(
        prompt,
        model_key="phi",
        max_new_tokens=256,
    )

    # build row for this example
    row = {
        "comment": original_comment,          # ground-truth
        "comment_created": slm_response,      # Phi output
        "model_name": "phi",
        "prompt_type": "zero_shot",

        # metrics
        "elapsed_s": metrics["elapsed_s"],
        "input_tokens": metrics["input_tokens"],
        "output_tokens": metrics["output_tokens"],
        "total_tokens": metrics["total_tokens"],
        "avg_power_w": metrics["avg_power_w"],
        "energy_joules": metrics["energy_joules"],
        "energy_wh": metrics["energy_wh"],
    }

    rows.append(row)

# Create DataFrame with one row per example
df_results = pd.DataFrame(rows)

# Optional peek
df_results.head()


In [ ]:
# Save to CSV
csv_path = "path.csv"
df_results.to_csv(csv_path, index=False)

# Automatically download the file
files.download(csv_path)
